# SC-15-Final-Project - Projet Final

**Navigation** : [Index](../README.md) | [<< Cross-Chain](../04-Multi-Chain/SC-14-Cross-Chain.ipynb)

---

## Projet : DApp Complete

Ce projet final combine tous les concepts appris :
- **Token ERC-20** avec mint/burn
- **Staking** avec rewards
- **Gouvernance** avec voting
- **Tests Foundry** complets

### Duree estimee : 90 minutes

---

## 1. Architecture du Projet

```
final-project/
|-- src/
|   |-- GovernanceToken.sol
|   |-- StakingContract.sol
|   |-- Governance.sol
|   `-- Treasury.sol
|-- test/
|   |-- GovernanceToken.t.sol
|   |-- Staking.t.sol
|   `-- Governance.t.sol
|-- script/
|   `-- Deploy.s.sol
`-- foundry.toml
```

---

## 2. Governance Token

In [1]:
# Governance Token
GOV_TOKEN = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@openzeppelin/contracts/token/ERC20/ERC20.sol";
import "@openzeppelin/contracts/token/ERC20/extensions/ERC20Burnable.sol";
import "@openzeppelin/contracts/access/Ownable.sol";
import "@openzeppelin/contracts/token/ERC20/extensions/ERC20Votes.sol";

contract GovernanceToken is ERC20, ERC20Burnable, Ownable, ERC20Votes {
    uint256 public constant MAX_SUPPLY = 1_000_000 * 10**18;

    constructor(address initialOwner)
        ERC20("Governance Token", "GOV")
        Ownable(initialOwner)
        ERC20Permit("Governance Token")
    {
        _mint(initialOwner, MAX_SUPPLY);
    }

    // Mint only by owner (for staking rewards)
    function mint(address to, uint256 amount) public onlyOwner {
        require(totalSupply() + amount <= MAX_SUPPLY, "Max supply exceeded");
        _mint(to, amount);
    }

    // Required overrides for ERC20Votes
    function _update(address from, address to, uint256 value)
        internal
        override(ERC20, ERC20Votes)
    {
        super._update(from, to, value);
    }

    function nonces(address owner)
        public
        view
        override(ERC20Permit, Nonces)
        returns (uint256)
    {
        return super.nonces(owner);
    }
}
'''

print("Governance Token:")
print(GOV_TOKEN)

Governance Token:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@openzeppelin/contracts/token/ERC20/ERC20.sol";
import "@openzeppelin/contracts/token/ERC20/extensions/ERC20Burnable.sol";
import "@openzeppelin/contracts/access/Ownable.sol";
import "@openzeppelin/contracts/token/ERC20/extensions/ERC20Votes.sol";

contract GovernanceToken is ERC20, ERC20Burnable, Ownable, ERC20Votes {
    uint256 public constant MAX_SUPPLY = 1_000_000 * 10**18;

    constructor(address initialOwner)
        ERC20("Governance Token", "GOV")
        Ownable(initialOwner)
        ERC20Permit("Governance Token")
    {
        _mint(initialOwner, MAX_SUPPLY);
    }

    // Mint only by owner (for staking rewards)
    function mint(address to, uint256 amount) public onlyOwner {
        require(totalSupply() + amount <= MAX_SUPPLY, "Max supply exceeded");
        _mint(to, amount);
    }

    // Required overrides for ERC20Votes
    function _update(address from, address to, uint256 value)
 

---

## 3. Staking Contract

In [2]:
# Staking Contract
STAKING_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@openzeppelin/contracts/token/ERC20/IERC20.sol";
import "@openzeppelin/contracts/token/ERC20/utils/SafeERC20.sol";
import "@openzeppelin/contracts/access/Ownable.sol";
import "@openzeppelin/contracts/utils/ReentrancyGuard.sol";

contract StakingContract is Ownable, ReentrancyGuard {
    using SafeERC20 for IERC20;

    IERC20 public immutable stakingToken;
    IERC20 public immutable rewardToken;

    uint256 public rewardRate = 100;  // tokens per block
    uint256 public lastUpdateTime;
    uint256 public rewardPerTokenStored;
    uint256 public totalStaked;

    mapping(address => uint256) public stakedBalance;
    mapping(address => uint256) public userRewardPerTokenPaid;
    mapping(address => uint256) public rewards;

    event Staked(address indexed user, uint256 amount);
    event Withdrawn(address indexed user, uint256 amount);
    event RewardClaimed(address indexed user, uint256 amount);

    constructor(
        address _stakingToken,
        address _rewardToken,
        address owner
    ) Ownable(owner) {
        stakingToken = IERC20(_stakingToken);
        rewardToken = IERC20(_rewardToken);
    }

    modifier updateReward(address account) {
        rewardPerTokenStored = rewardPerToken();
        lastUpdateTime = block.timestamp;
        if (account != address(0)) {
            rewards[account] = earned(account);
            userRewardPerTokenPaid[account] = rewardPerTokenStored;
        }
        _;
    }

    function rewardPerToken() public view returns (uint256) {
        if (totalStaked == 0) return rewardPerTokenStored;
        return rewardPerTokenStored + (
            (block.timestamp - lastUpdateTime) * rewardRate * 1e18 / totalStaked
        );
    }

    function earned(address account) public view returns (uint256) {
        return stakedBalance[account] * (
            rewardPerToken() - userRewardPerTokenPaid[account]
        ) / 1e18 + rewards[account];
    }

    function stake(uint256 amount) external nonReentrant updateReward(msg.sender) {
        require(amount > 0, "Amount must be > 0");

        stakingToken.safeTransferFrom(msg.sender, address(this), amount);
        stakedBalance[msg.sender] += amount;
        totalStaked += amount;

        emit Staked(msg.sender, amount);
    }

    function withdraw(uint256 amount) external nonReentrant updateReward(msg.sender) {
        require(amount > 0, "Amount must be > 0");
        require(stakedBalance[msg.sender] >= amount, "Insufficient balance");

        stakedBalance[msg.sender] -= amount;
        totalStaked -= amount;
        stakingToken.safeTransfer(msg.sender, amount);

        emit Withdrawn(msg.sender, amount);
    }

    function claimReward() external nonReentrant updateReward(msg.sender) {
        uint256 reward = rewards[msg.sender];
        require(reward > 0, "No rewards");

        rewards[msg.sender] = 0;
        rewardToken.safeTransfer(msg.sender, reward);

        emit RewardClaimed(msg.sender, reward);
    }

    function setRewardRate(uint256 _rate) external onlyOwner {
        rewardPerTokenStored = rewardPerToken();
        lastUpdateTime = block.timestamp;
        rewardRate = _rate;
    }
}
'''

print("Staking Contract:")
print(STAKING_CONTRACT)

Staking Contract:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@openzeppelin/contracts/token/ERC20/IERC20.sol";
import "@openzeppelin/contracts/token/ERC20/utils/SafeERC20.sol";
import "@openzeppelin/contracts/access/Ownable.sol";
import "@openzeppelin/contracts/utils/ReentrancyGuard.sol";

contract StakingContract is Ownable, ReentrancyGuard {
    using SafeERC20 for IERC20;

    IERC20 public immutable stakingToken;
    IERC20 public immutable rewardToken;

    uint256 public rewardRate = 100;  // tokens per block
    uint256 public lastUpdateTime;
    uint256 public rewardPerTokenStored;
    uint256 public totalStaked;

    mapping(address => uint256) public stakedBalance;
    mapping(address => uint256) public userRewardPerTokenPaid;
    mapping(address => uint256) public rewards;

    event Staked(address indexed user, uint256 amount);
    event Withdrawn(address indexed user, uint256 amount);
    event RewardClaimed(address indexed user, uint256 amount);

  

---

## 4. Governance Contract

In [3]:
# Governance Contract
GOVERNANCE_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@openzeppelin/contracts/governance/Governor.sol";
import "@openzeppelin/contracts/governance/extensions/GovernorVotes.sol";
import "@openzeppelin/contracts/governance/extensions/GovernorVotesQuorumFraction.sol";
import "@openzeppelin/contracts/governance/extensions/GovernorTimelockControl.sol";
import "@openzeppelin/contracts/governance/TimelockController.sol";

contract MyGovernor is
    Governor,
    GovernorVotes,
    GovernorVotesQuorumFraction,
    GovernorTimelockControl
{
    uint256 private constant VOTING_DELAY = 1;       // 1 block
    uint256 private constant VOTING_PERIOD = 45818;  // ~1 week
    uint256 private constant PROPOSAL_THRESHOLD = 0; // anyone can propose

    constructor(
        IVotes _token,
        TimelockController _timelock
    )
        Governor("MyGovernor")
        GovernorVotes(_token)
        GovernorVotesQuorumFraction(4)  // 4% quorum
        GovernorTimelockControl(_timelock)
    {}

    function votingDelay() public pure override returns (uint256) {
        return VOTING_DELAY;
    }

    function votingPeriod() public pure override returns (uint256) {
        return VOTING_PERIOD;
    }

    function proposalThreshold() public pure override returns (uint256) {
        return PROPOSAL_THRESHOLD;
    }

    // Required overrides
    function state(uint256 proposalId)
        public
        view
        override(Governor, GovernorTimelockControl)
        returns (ProposalState)
    {
        return super.state(proposalId);
    }

    function _executeOperations(
        uint256 proposalId,
        address[] memory targets,
        uint256[] memory values,
        bytes[] memory calldatas,
        bytes32 descriptionHash
    ) internal override(Governor, GovernorTimelockControl) {
        super._executeOperations(proposalId, targets, values, calldatas, descriptionHash);
    }

    function _cancel(
        address[] memory targets,
        uint256[] memory values,
        bytes[] memory calldatas,
        bytes32 descriptionHash
    ) internal override(Governor, GovernorTimelockControl) returns (uint256) {
        return super._cancel(targets, values, calldatas, descriptionHash);
    }

    function _executor()
        internal
        view
        override(Governor, GovernorTimelockControl)
        returns (address)
    {
        return super._executor();
    }
}
'''

print("Governance Contract:")
print(GOVERNANCE_CONTRACT)

Governance Contract:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@openzeppelin/contracts/governance/Governor.sol";
import "@openzeppelin/contracts/governance/extensions/GovernorVotes.sol";
import "@openzeppelin/contracts/governance/extensions/GovernorVotesQuorumFraction.sol";
import "@openzeppelin/contracts/governance/extensions/GovernorTimelockControl.sol";
import "@openzeppelin/contracts/governance/TimelockController.sol";

contract MyGovernor is
    Governor,
    GovernorVotes,
    GovernorVotesQuorumFraction,
    GovernorTimelockControl
{
    uint256 private constant VOTING_DELAY = 1;       // 1 block
    uint256 private constant VOTING_PERIOD = 45818;  // ~1 week
    uint256 private constant PROPOSAL_THRESHOLD = 0; // anyone can propose

    constructor(
        IVotes _token,
        TimelockController _timelock
    )
        Governor("MyGovernor")
        GovernorVotes(_token)
        GovernorVotesQuorumFraction(4)  // 4% quorum
        GovernorTimelockCon

---

## 5. Tests Foundry

In [4]:
# Tests Foundry
FOUNDRY_TESTS = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "forge-std/Test.sol";
import "../src/GovernanceToken.sol";
import "../src/StakingContract.sol";

contract StakingTest is Test {
    GovernanceToken token;
    StakingContract staking;
    address owner = address(1);
    address user1 = address(2);
    address user2 = address(3);

    function setUp() public {
        vm.startPrank(owner);
        token = new GovernanceToken(owner);
        staking = new StakingContract(
            address(token),
            address(token),
            owner
        );
        token.transferOwnership(address(staking));
        vm.stopPrank();

        // Distribute tokens
        vm.prank(owner);
        token.transfer(user1, 1000 * 10**18);
        vm.prank(owner);
        token.transfer(user2, 1000 * 10**18);
    }

    function test_Stake() public {
        vm.startPrank(user1);
        token.approve(address(staking), 100 * 10**18);
        staking.stake(100 * 10**18);

        assertEq(staking.stakedBalance(user1), 100 * 10**18);
        assertEq(staking.totalStaked(), 100 * 10**18);
        vm.stopPrank();
    }

    function test_Withdraw() public {
        vm.startPrank(user1);
        token.approve(address(staking), 100 * 10**18);
        staking.stake(100 * 10**18);

        staking.withdraw(50 * 10**18);
        assertEq(staking.stakedBalance(user1), 50 * 10**18);
        vm.stopPrank();
    }

    function testFuzz_Stake(uint256 amount) public {
        vm.assume(amount > 0 && amount <= 1000 * 10**18);

        vm.startPrank(user1);
        token.approve(address(staking), amount);
        staking.stake(amount);

        assertEq(staking.stakedBalance(user1), amount);
        vm.stopPrank();
    }

    function test_Rewards() public {
        vm.startPrank(user1);
        token.approve(address(staking), 100 * 10**18);
        staking.stake(100 * 10**18);

        // Advance time
        vm.warp(block.timestamp + 1 days);

        uint256 reward = staking.earned(user1);
        assertGt(reward, 0);
        vm.stopPrank();
    }
}
'''

print("Tests Foundry:")
print(FOUNDRY_TESTS)

Tests Foundry:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "forge-std/Test.sol";
import "../src/GovernanceToken.sol";
import "../src/StakingContract.sol";

contract StakingTest is Test {
    GovernanceToken token;
    StakingContract staking;
    address owner = address(1);
    address user1 = address(2);
    address user2 = address(3);

    function setUp() public {
        vm.startPrank(owner);
        token = new GovernanceToken(owner);
        staking = new StakingContract(
            address(token),
            address(token),
            owner
        );
        token.transferOwnership(address(staking));
        vm.stopPrank();

        // Distribute tokens
        vm.prank(owner);
        token.transfer(user1, 1000 * 10**18);
        vm.prank(owner);
        token.transfer(user2, 1000 * 10**18);
    }

    function test_Stake() public {
        vm.startPrank(user1);
        token.approve(address(staking), 100 * 10**18);
        staking.stake(100 * 10**18);

---

## 6. Script de Deploiement

In [5]:
# Deploy Script
DEPLOY_SCRIPT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "forge-std/Script.sol";
import "../src/GovernanceToken.sol";
import "../src/StakingContract.sol";
import "@openzeppelin/contracts/governance/TimelockController.sol";
import "../src/MyGovernor.sol";

contract Deploy is Script {
    function run() external {
        uint256 deployerPrivateKey = vm.envUint("PRIVATE_KEY");
        vm.startBroadcast(deployerPrivateKey);

        // Deploy token
        GovernanceToken token = new GovernanceToken(msg.sender);

        // Deploy staking
        StakingContract staking = new StakingContract(
            address(token),
            address(token),
            msg.sender
        );

        // Deploy timelock
        address[] memory proposers = new address[](1);
        proposers[0] = msg.sender;
        address[] memory executors = new address[](1);
        executors[0] = address(0);  // Anyone can execute

        TimelockController timelock = new TimelockController(
            1 days,  // min delay
            proposers,
            executors,
            msg.sender  // admin
        );

        // Deploy governor
        MyGovernor governor = new MyGovernor(
            IVotes(address(token)),
            timelock
        );

        // Setup roles
        bytes32 proposerRole = timelock.PROPOSER_ROLE();
        bytes32 executorRole = timelock.EXECUTOR_ROLE();
        bytes32 adminRole = timelock.TIMELOCK_ADMIN_ROLE();

        timelock.grantRole(proposerRole, address(governor));
        timelock.grantRole(executorRole, address(0));
        timelock.revokeRole(adminRole, msg.sender);

        vm.stopBroadcast();

        // Log addresses
        console.log("Token:", address(token));
        console.log("Staking:", address(staking));
        console.log("Timelock:", address(timelock));
        console.log("Governor:", address(governor));
    }
}
'''

print("Script de Deploiement:")
print(DEPLOY_SCRIPT)

Script de Deploiement:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "forge-std/Script.sol";
import "../src/GovernanceToken.sol";
import "../src/StakingContract.sol";
import "@openzeppelin/contracts/governance/TimelockController.sol";
import "../src/MyGovernor.sol";

contract Deploy is Script {
    function run() external {
        uint256 deployerPrivateKey = vm.envUint("PRIVATE_KEY");
        vm.startBroadcast(deployerPrivateKey);

        // Deploy token
        GovernanceToken token = new GovernanceToken(msg.sender);

        // Deploy staking
        StakingContract staking = new StakingContract(
            address(token),
            address(token),
            msg.sender
        );

        // Deploy timelock
        address[] memory proposers = new address[](1);
        proposers[0] = msg.sender;
        address[] memory executors = new address[](1);
        executors[0] = address(0);  // Anyone can execute

        TimelockController timelock = new Timeloc

---

## 7. Commandes de Deploiement

In [6]:
# Commandes
print("""
COMMANDES DEPLOIEMENT

# Installation
forge install OpenZeppelin/openzeppelin-contracts --no-commit
forge install foundry-rs/forge-std --no-commit

# Compilation
forge build

# Tests
forge test -vvv
forge test --fuzz-runs 1000
forge coverage

# Deploiement local
anvil
forge script script/Deploy.s.sol --rpc-url http://localhost:8545 --broadcast

# Deploiement Sepolia
forge script script/Deploy.s.sol \
    --rpc-url $SEPOLIA_RPC_URL \
    --private-key $PRIVATE_KEY \
    --etherscan-api-key $ETHERSCAN_API_KEY \
    --broadcast --verify

# Verification
forge verify-contract <address> GovernanceToken \
    --chain sepolia \
    --etherscan-api-key $ETHERSCAN_API_KEY
""")


COMMANDES DEPLOIEMENT

# Installation
forge install OpenZeppelin/openzeppelin-contracts --no-commit
forge install foundry-rs/forge-std --no-commit

# Compilation
forge build

# Tests
forge test -vvv
forge test --fuzz-runs 1000
forge coverage

# Deploiement local
anvil
forge script script/Deploy.s.sol --rpc-url http://localhost:8545 --broadcast

# Deploiement Sepolia
forge script script/Deploy.s.sol     --rpc-url $SEPOLIA_RPC_URL     --private-key $PRIVATE_KEY     --etherscan-api-key $ETHERSCAN_API_KEY     --broadcast --verify

# Verification
forge verify-contract <address> GovernanceToken     --chain sepolia     --etherscan-api-key $ETHERSCAN_API_KEY



---

## 8. Checklist Finale

Avant de deploier en production :

- [ ] Tests unitaires passent (100% coverage)
- [ ] Fuzz tests passent
- [ ] Audit de code effectue
- [ ] Variables d'environnement configurees
- [ ] Adresse owner correcte (multisig)
- [ ] Limite de gas verifiee
- [ ] Events documentes
- [ ] Verification Etherscan prete
- [ ] Plan de mise a jour defini
- [ ] Monitoring configure

---

**Felicitation!** Vous avez complete le cours Smart Contracts.

**Retour** : [Index](../README.md)

## Exercice Final : DApp Complete de A a Z

Concevez et deployez une DApp complete avec tous les concepts appris.

### Objectifs
1. Creer une architecture multi-contrats coherente
2. Implementer les tests complets (unit + fuzz + integration)
3. Deployer sur testnet avec verification
4. Documenter le projet

### Instructions

#### Phase 1 : Conception (30 min)
```markdown
# TODO: Design votre DApp
## Nom: [Votre choix]
## Fonctionnalites:
1. [Feature 1]
2. [Feature 2]
3. [Feature 3]

## Contrats necessaires:
- ContractA.sol: [description]
- ContractB.sol: [description]

## Interactions:
[Diagramme ou description des appels entre contrats]
```

#### Phase 2 : Implementation (45 min)
```solidity
// TODO: Implementez vos contrats dans src/
// - Suivez les patterns OpenZeppelin
// - Ajoutez des events pour tracking
// - N'oubliez pas ReentrancyGuard et Ownable
// - Commentez les fonctions publiquess
```

#### Phase 3 : Tests (30 min)
```solidity
// TODO: test/[Contract].t.sol
// Tests obligatoires:
// - test_[feature principale]
// - test_[cas d'erreur]
// - testFuzz_[parametres variables]
// - test_[access control]
// - test_[edge cases]
```

#### Phase 4 : Deploiement (15 min)
```bash
# TODO: Deployez sur Sepolia
forge script script/Deploy.s.sol \
    --rpc-url $SEPOLIA_RPC_URL \
    --private-key $PRIVATE_KEY \
    --etherscan-api-key $ETHERSCAN_API_KEY \
    --broadcast --verify

# TODO: Verifiez sur Etherscan
# TODO: Testez les fonctions via l'interface Etherscan
```

### Checklist de validation
- [ ] Architecture documentee
- [ ] Contrats compilent sans warnings
- [ ] Tests passent a 100%
- [ ] Fuzz tests passent (1000 runs)
- [ ] Coverage > 90%
- [ ] Deploiement Sepolia reussi
- [ ] Verification Etherscan completee
- [ ] README avec instructions d'utilisation

### Bonus
- Ajouter une interface frontend (React + wagmi)
- Implementer upgradability (UUPS proxy)
- Ajouter integration Chainlink (VRF ou Price Feed)